In [4]:
!pip install pymysql
#디비 연결을 위해 필요한 패키지, 한번만 실행

In [1]:
#행렬과 관련된 수학식을 사용할 수 있는 라이브러리
import numpy as np

#데이터분석용 pandas 사용을 위한 라이브러리
import pandas as pd

#데이터베이스 연결 위한 라이브러리
from pymysql import connect, cursors

#그래프, 시각화를 위한 라이브러리
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

#문자데이터를 숫자로 인코딩하기 위한 라이브러리
from sklearn.preprocessing import LabelEncoder

#전체 데이터를 학습용 데이터와 검증용 데이터로 나누기 위한 라이브러리
from sklearn.model_selection import train_test_split

#분석 평가 지표 사용을 위한 라이브러리
from sklearn.metrics import  mean_absolute_error, mean_squared_error, r2_score

In [2]:
fm._load_fontmanager(try_read_cache=False)
plt.rc("font", family="NanumGothic")
plt.rc("axes", unicode_minus=False)

In [3]:
conn = connect(
    host='192.168.30.79',       
    port=3306,
    user='user',
    password='mariadb',
    database='project-2',
    charset='utf8mb4'       # 한글 깨짐 방지
)

cursor = conn.cursor()

cursor.execute("SELECT 1")
print(cursor.fetchone())    # (1,) 이 나오면 연결 성공

(1,)


In [4]:
df = pd.read_csv('가스_인구_기온_통합.csv',  encoding='cp949')
df

,연월,시도,월평균기온,월평균최고기온,월평균최저기온,가구수(개),공급량(㎥),총인구수
0,2021-01,강원,-4.2,1.5,-9.7,386599,51918.194190,1541696
1,2021-02,강원,1.3,7.3,-4.6,387774,43068.962620,1540875
2,2021-03,강원,7.0,13.3,1.3,388403,32782.803000,1536399
3,2021-04,강원,11.8,17.9,5.7,390610,20762.350470,1536175
4,2021-05,강원,15.3,21.1,10.0,391098,14472.110240,1535491
...,...,...,...,...,...,...,...,...
1183,2026-02,충북,1.1,7.7,-5.0,584666,47598.860310,1596816
1184,2026-03,충북,6.8,13.6,0.5,579452,39099.270910,1597976
1185,2026-04,충북,13.5,20.6,6.4,576171,21893.604080,1599375
1186,2026-05,충북,18.6,25.5,11.8,576065,12157.524940,1600216


In [5]:
cursor = conn.cursor()

# region 테이블에서 (번호, 이름) 전부 가져오기
cursor.execute("SELECT region_id, region_name FROM region")
rows = cursor.fetchall()          # ((1, '서울'), (2, '부산'), ...)

# 변환표 만들기: {'서울': 1, '부산': 2, ...}
region_map = {}
for rid, name in rows:
    region_map[name] = rid
print(region_map)

{'강원': 10, '경기': 9, '경남': 16, '경북': 15, '광주': 5, '대구': 3, '대전': 6, '부산': 2, '서울': 1, '세종': 8, '울산': 7, '인천': 4, '전국': 18, '전남': 14, '전북': 13, '제주': 17, '충남': 12, '충북': 11}


In [7]:
# '시도' 컬럼의 이름을 변환표로 번호로 바꿔서 새 컬럼에 저장
df['region_id'] = df['시도'].map(region_map)

# ★ 꼭 확인: 변환표에 없는 이름이 있으면 region_id 가 비어(NaN) 있음
print(df[df['region_id'].isna()]['시도'].unique())  # [] 이면 전부 매칭 성공
df


[]


,연월,시도,월평균기온,월평균최고기온,월평균최저기온,가구수(개),공급량(㎥),총인구수,region_id
0,2021-01,강원,-4.2,1.5,-9.7,386599,51918.194190,1541696,10
1,2021-02,강원,1.3,7.3,-4.6,387774,43068.962620,1540875,10
2,2021-03,강원,7.0,13.3,1.3,388403,32782.803000,1536399,10
3,2021-04,강원,11.8,17.9,5.7,390610,20762.350470,1536175,10
4,2021-05,강원,15.3,21.1,10.0,391098,14472.110240,1535491,10
...,...,...,...,...,...,...,...,...,...
1183,2026-02,충북,1.1,7.7,-5.0,584666,47598.860310,1596816,11
1184,2026-03,충북,6.8,13.6,0.5,579452,39099.270910,1597976,11
1185,2026-04,충북,13.5,20.6,6.4,576171,21893.604080,1599375,11
1186,2026-05,충북,18.6,25.5,11.8,576065,12157.524940,1600216,11


In [8]:
cursor = conn.cursor()
insert_sql = "INSERT INTO GAS_MONTHLY (REGION_ID, YM, AVG_TEMP, MAX_TEMP, MIN_TEMP, HOUSEHOLD_CNT, POPULATION, SUPPLY) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"
insert_param = []
for e in df.values:
  each_data = (e[8], e[0], e[2], e[3], e[4], e[5], e[7], e[6])
  insert_param.append(each_data)

#쿼리 실행
cursor.executemany(insert_sql, insert_param)
#커밋
conn.commit()